# Read policy signs from rendered observations

**Evidence state:** Defined

## What this demonstrates

Render a bounded arcade state as pixels, address a calibrated visual index, and recover the corresponding policy action through `VisualSignReader`.

## Why it matters

Observation, visual addressing, policy storage, and action selection remain separate and inspectable.


In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "VERSION").is_file():
    if ROOT.parent == ROOT:
        raise RuntimeError("ZeroModel repository root not found")
    ROOT = ROOT.parent
os.chdir(ROOT)
for path in (ROOT, ROOT / "examples"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
print(f"repository root: {ROOT}")


repository root: C:\Projects\zeromodel


## Source and package mapping

- `examples/arcade_visual_sign_reader.py`
- `examples/arcade_shooter_policy.py`
- `zeromodel-vision`
- `zeromodel-video`


In [2]:
import json
from IPython.display import Image, display
from examples.arcade_visual_sign_reader import ShooterConfig, compile_policy_artifact, compile_visual_index_artifact, make_visual_reader, render_state_frame, run_visual_policy_episode
from zeromodel.core import png_bytes

config = ShooterConfig()
policy = compile_policy_artifact(config)
visual_build = compile_visual_index_artifact(config, policy_artifact=policy)
reader = make_visual_reader(policy, visual_build)
frame = render_state_frame(0, 2, 0, width=config.width)
display(Image(data=png_bytes(frame.astype("float64") / 255.0), width=420))
decision = reader.read(frame)
print(json.dumps({"policy_artifact_id": policy.artifact_id, "visual_index_artifact_id": visual_build.artifact.artifact_id, "decision": decision.to_dict()}, indent=2, sort_keys=True))


{
  "decision": {
    "acceptance_threshold": 0.5,
    "accepted": true,
    "action": "RIGHT",
    "calibration_digest": "534f82ed6375bd7ff3a9cdc28ebfe1bf6345e19e98c1ab28fcdc1509139fee4f",
    "candidates": {
      "FIRE": 0.0,
      "LEFT": 0.0,
      "RIGHT": 1.0,
      "STAY": 0.1
    },
    "distance_margin": 2.0,
    "evidence": {},
    "exact_feature_match": true,
    "feature_digest": "sha256:d4614689ffd5aea7241ef455a55c1b8387ea6a902943376cbef545a4905bea5e",
    "feature_spec_digest": "942fb6db1968412b60a30b404b402ef7fa6b1de65464d5839d1a225337a8634f",
    "input_digest": "sha256:f50689748dc92a818254c73bbcc414734d33466de9c8e0cc2a30403a80843a5c",
    "matched_row_id": "tank=0|target=2|cooldown=0",
    "nearest_distance": 0.0,
    "nearest_row_id": "tank=0|target=2|cooldown=0",
    "policy_artifact_id": "eb7523f406b45ac30b478fe9528db8f89a548693b0add2fc8d3e51c4badd857e",
    "reader_version": "zeromodel-visual-sign-reader/v2",
    "reason": "accepted",
    "required_margin": 1.5,
 

## Application

```text
rendered observation -> feature transform -> calibrated visual index -> policy row -> action plus evidence
```


In [3]:
episode = run_visual_policy_episode(config, policy_artifact=policy, visual_index_build=visual_build, visual_reader=reader)
print(json.dumps({"cleared": episode["cleared"], "score": episode["score"], "steps": episode["steps"], "first_decision": episode["trace"][0]["visual_decision"]}, indent=2))


{
  "cleared": true,
  "score": 4,
  "steps": 22,
  "first_decision": {
    "accepted": true,
    "reason": "accepted",
    "input_digest": "sha256:fb2df1df4602f32c8750733f518689d6c7db4114deb940572716bb00310921c6",
    "feature_digest": "sha256:a32afc31b51207442d8660863365b04d9a3aebbd22139ce2d35ebc40989a8c24",
    "reader_version": "zeromodel-visual-sign-reader/v2",
    "visual_index_artifact_id": "9362cdfec268a99994cb1d84b16f30fac7519d5cb2899ef02ec7e1f737172650",
    "policy_artifact_id": "eb7523f406b45ac30b478fe9528db8f89a548693b0add2fc8d3e51c4badd857e",
    "feature_spec_digest": "942fb6db1968412b60a30b404b402ef7fa6b1de65464d5839d1a225337a8634f",
    "calibration_digest": "534f82ed6375bd7ff3a9cdc28ebfe1bf6345e19e98c1ab28fcdc1509139fee4f",
    "nearest_row_id": "tank=3|target=0|cooldown=0",
    "nearest_distance": 0.0,
    "second_nearest_row_id": "tank=3|target=0|cooldown=1",
    "second_nearest_distance": 2.0,
    "distance_margin": 2.0,
    "acceptance_threshold": 0.5,
    "requir

## Boundaries and limitations

The arcade fixture provides bounded symbolic ground truth. This fast notebook does not run the exhaustive sweep and does not establish robustness to unseen or open-world imagery.

## Reproduction record

The builder records execution metadata and HTML under `docs/results/demos/visual-sign-reader/`.


In [4]:
print(json.dumps({"demo_id": "visual-sign-reader", "policy_artifact_id": policy.artifact_id, "visual_index_artifact_id": visual_build.artifact.artifact_id}, indent=2))


{
  "demo_id": "visual-sign-reader",
  "policy_artifact_id": "eb7523f406b45ac30b478fe9528db8f89a548693b0add2fc8d3e51c4badd857e",
  "visual_index_artifact_id": "9362cdfec268a99994cb1d84b16f30fac7519d5cb2899ef02ec7e1f737172650"
}
